## FLOOD DETECTION SCENARIO PIPELINE

In [1]:
import digitalhub as dh
PROJECT_NAME = "flood-detection"
proj = dh.get_or_create_project(PROJECT_NAME) 

### Download Sentinel Data

Register to the open data space copernicus(if not already) and get your credentials.

https://identity.dataspace.copernicus.eu/auth/realms/CDSE/login-actions/registration?client_id=cdse-public&tab_id=FIiRPJeoiX4

Log the credentials as project secret keys as shown below

In [ ]:
# THIS NEED TO BE EXECUTED JUST ONCE
secret0 = proj.new_secret(name="CDSETOOL_ESA_USER", secret_value="esa_username")
secret1 = proj.new_secret(name="CDSETOOL_ESA_PASSWORD", secret_value="esa_password")

### Function Download Sentinel 1 / 2

In [ ]:
function_s2 = proj.new_function("download-images-s2",kind="container",image="ghcr.io/tn-aixpa/sentinel-tools:0.11.6",command="python")

In [ ]:
function_s1 = proj.new_function("download-images-s1",kind="container",image="ghcr.io/tn-aixpa/sentinel-tools:0.11.6",command="python")

### Log artifact

The pipeline requires shape files input of river, lakes, and slope.

Log the river shape file. Download the zip file from the [SIAT Portal](https://siat.provincia.tn.it/geonetwork/srv/ita/catalog.search#/metadata/p_TN:df06e63c-d0f3-46c9-8ec2-c25a22c50ef7) and extract the contents inside a folder 'Rivers_TN' and log it as project artifact

In [ ]:
artifact_name='Rivers_TN'
src_path='Rivers_TN'
artifact_bosco = proj.log_artifact(name=artifact_name, kind="artifact", source=src_path)

Log the lakes shape file. Download the zip file from the [SIAT Portal](https://siat.provincia.tn.it/geonetwork/srv/ita/catalog.search#/metadata/p_TN:0f1fdc33-5c71-4c6d-81e7-25eb2ab0e599) and extract the contents inside a folder 'Lakes_TN' and log it as project artifact

In [ ]:
artifact_name='Lakes_TN'
src_path='Lakes_TN'
artifact_bosco = proj.log_artifact(name=artifact_name, kind="artifact", source=src_path)

Log the slope shape file. Download the zip file from the [Huggingface repository](https://huggingface.co/datasets/lbergamasco/trentino-slope-map/blob/main/trentino_slope_map.tif) and extract the contents inside a folder 'Slopes_TN' and log it as project artifact

In [ ]:
artifact_name='Slopes_TN'
src_path='Slopes_TN'
artifact_bosco = proj.log_artifact(name=artifact_name, kind="artifact", source=src_path)

The resulting datasets will be registered as the project artifact in the datalake under the given names ('Rivers_TN', 'Slopes_TN', 'Lakes_TN').

### Elaboration

In [ ]:
function_rs = proj.new_function("elaborate",kind="container", image="ghcr.io/tn-aixpa/rs-flood-mapping:3.1", code_src="launch.sh")

### Pipeline (KFP)

In [ ]:
%%writefile "flood_pipeline.py"

from digitalhub_runtime_kfp.dsl import pipeline_context
import datetime

def myhandler(geometry, outputName, floodDate, aoiName, s1_preFloodDate, s1_postFloodDate, s2_preFloodDate, s2_postFloodDate):
  
    s1_artifact_pre = "sentinel1_GRD_preflood_" + str(outputName)
    s1_artifact_post = "sentinel1_GRD_postflood_"+ str(outputName) 
    s2_artifact_pre = "sentinel2_pre_flood_"+ str(outputName)
    s2_artifact_post =  "sentinel2_post_flood_"+ str(outputName)
    
    string_dict_data_s1Pre =  """{"satelliteParams": {"satelliteType": "Sentinel1","processingLevel": "LEVEL1","sensorMode": "IW","productType": "GRD"},"startDate":\"""" + str(s1_preFloodDate) + """\","endDate": \"""" + str(floodDate) + """\","geometry": \"""" + str(geometry) + """\","area_sampling": "True","tmp_path_same_folder_dwl":"True","artifact_name":  \"""" + str(s1_artifact_pre) + """\"}"""
    string_dict_data_s1Post = """{"satelliteParams": {"satelliteType": "Sentinel1","processingLevel": "LEVEL1","sensorMode": "IW","productType": "GRD"},"startDate":\"""" + str(floodDate) + """\","endDate": \"""" + str(s1_postFloodDate) + """\","geometry": \"""" + str(geometry) + """\","area_sampling": "True","tmp_path_same_folder_dwl":"True","artifact_name": \"""" + str(s1_artifact_post) + """\"}"""
    string_dict_data_s2Pre =  """{"satelliteParams":{"satelliteType": "Sentinel2","processingLevel": "S2MSI2A","bandmath": ["NDWI"]},"startDate":\"""" + str(s2_preFloodDate) + """\","endDate": \"""" + str(floodDate) + """\","geometry": \"""" + str(geometry) + """\","cloudCover": "[0,20]","area_sampling": "True","artifact_name" : \"""" + str(s2_artifact_pre) + """\","preprocess_data_only": "false"}"""
    string_dict_data_s2Post = """{"satelliteParams":{"satelliteType": "Sentinel2","processingLevel": "S2MSI2A","bandmath": ["NDWI"]},"startDate":\"""" + str(floodDate) + """\","endDate": \"""" + str(s2_postFloodDate) + """\","geometry": \"""" + str(geometry) + """\","cloudCover": "[0,20]","area_sampling": "True","artifact_name": \"""" + str(s2_artifact_post) + """\","preprocess_data_only": "false"}"""

    
    
    with pipeline_context() as pc:

        s1 = pc.step(name="downloadS1Pre",
                     function="download-images-s1",
                     action="job",
                     secrets=["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                     fs_group='8877',
                     args=["main.py", string_dict_data_s1Pre],
                     resources={"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/files",
                        "spec": { "size": "100Gi" }
                        }
                    ])

        s2 = pc.step(name="downloadS1Post",
                     function="download-images-s1",
                     action="job",
                     secrets=["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                     fs_group='8877',
                     args=["main.py", string_dict_data_s1Post],
                     resources={"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/files",
                        "spec": { "size": "100Gi" }
                        }
                    ]).after(s1)
        
        s3 = pc.step(name="downloadS2Pre",
                     function="download-images-s2",
                     action="job",
                     secrets=["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                     fs_group='8877',
                     args=["main.py", string_dict_data_s2Pre],
                     resources={"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/files",
                        "spec": { "size": "100Gi" }
                        }
                    ]).after(s2)

        s4 = pc.step(name="downloadS2Post",
                     function="download-images-s2",
                     action="job",
                     secrets=["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                     fs_group='8877',
                     args=["main.py", string_dict_data_s2Post],
                     resources={"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/files",
                        "spec": { "size": "100Gi" }
                        }
                    ]).after(s3)

        s5 = pc.step(name="elaborate",
                     function="elaborate",
                     action="job",
                     fs_group='8877',
                     resources={"cpu": {"requests": "3", "limits": "6"},"mem":{"requests": "32Gi", "limits": "64Gi"}},
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-flood",
                        "mount_path": "/app/data",
                        "spec": { "size": "200Gi" }
                    }],
                     args=['/shared/launch.sh', str(s1_artifact_pre), str(s1_artifact_post), str(s2_artifact_pre), str(s2_artifact_post), str(geometry), 'Slopes_TN', 'trentino_slope_map.tif', 'Lakes_TN', 'idrspacq.shp', 'Rivers_TN', 'cif_pta2022_v.shp', str(outputName), str(floodDate), 'EPSG:25832', "['VV','VH']", '900', '17', '9', '4', str(aoiName)]
                     ).after(s4)
     


Overwriting flood_pipeline.py


Create workflow using project repo source file

In [ ]:
workflow = proj.new_workflow(
name="pipeline_flood_gitversion",
kind="kfp",
code_src="git+https://<username>:<personal_access_token>@github.com/tn-aixpa/rs-flood-mapping",
handler="src.flood_pipeline:myhandler")

(Optional) In case of forseen changes, one can create the workflow locally after modifying the code as shown in cell above.

In [ ]:
workflow = proj.new_workflow(name="pipeline_flood", kind="kfp", code_src= "flood_pipeline.py", handler = "myhandler")

Build workflow

In [ ]:
wfbuild = workflow.run(action="build", wait=True)

Run workflow (Garda)

In [ ]:
workflow_run = workflow.run(action="pipeline", parameters={
    "geometry":"POLYGON ((10.644988646837982 45.85539621678084, 10.644988646837982 46.06780100571985, 10.991744628283294 46.06780100571985, 10.991744628283294 45.85539621678084, 10.644988646837982 45.85539621678084))",
    "outputName": "garda_oct_2020",
    "floodDate":"2020-10-02",
    "aoiName": "garda",
    "s1_preFloodDate": "2020-09-25",
    "s1_postFloodDate": "2020-10-09",
    "s2_preFloodDate": "2020-09-12",
    "s2_postFloodDate": "2020-10-22"
    })

Run workflow (Vad di Non)

In [ ]:
workflow_run = workflow.run(action="pipeline", parameters={
    "geometry":"POLYGON ((10.81356293600004 46.09024746800003, 10.81356293600004 46.56319437500008, 11.285659684000052 46.56319437500008, 11.285659684000052 46.09024746800003, 10.81356293600004 46.09024746800003))",
    "outputName": "val_di_non_oct_2018",
    "floodDate":"2018-10-27",
    "aoiName": "val di non",
    "s1_preFloodDate": "2018-10-20",
    "s1_postFloodDate": "2018-11-04",
    "s2_preFloodDate": "2018-10-07",
    "s2_postFloodDate": "2018-11-17"
    })

Run workflow (Val di Fassa)

In [ ]:
workflow_run = workflow.run(action="pipeline", parameters={
    "geometry":"POLYGON ((11.123570406000056 45.89602853500003, 11.123570406000056 46.35080675000006, 11.833650907000049 46.35080675000006, 11.833650907000049 45.89602853500003, 11.123570406000056 45.89602853500003))",
    "outputName": "val_di_fassa_oct_2018",
    "floodDate":"2018-10-27",
    "aoiName": "val di fassa",
    "s1_preFloodDate": "2018-10-20",
    "s1_postFloodDate": "2018-11-04",
    "s2_preFloodDate": "2018-10-07",
    "s2_postFloodDate": "2018-11-17"
    })

### Pipeline (HERA-Argos Framework)

In [ ]:
%%writefile "flood_pipeline_hera.py"

from hera.workflows import Workflow, DAG, Parameter
from digitalhub_runtime_hera.dsl import step

def pipeline():
    # Create a new Workflow with an entrypoint DAG and a parameter
    with Workflow(entrypoint="dag", arguments=[
        Parameter(name="geometry"),
        Parameter(name="outputName"),
        Parameter(name="floodDate"),
        Parameter(name="aoiName"),
        Parameter(name="s1_preFloodDate"),
        Parameter(name="s1_postFloodDate"),
        Parameter(name="s2_preFloodDate"),
        Parameter(name="s2_postFloodDate"),
        ]) as w:

        with DAG(name="dag"):
            # Create a new Workflow with an entrypoint DAG and a parameter
            s1_artifact_pre = "sentinel1_GRD_preflood_" + str(w.get_parameter("outputName"))
            s1_artifact_post = "sentinel1_GRD_postflood_"+ str(w.get_parameter("outputName"))
            s2_artifact_pre = "sentinel2_pre_flood_"+ str(w.get_parameter("outputName"))
            s2_artifact_post =  "sentinel2_post_flood_"+ str(w.get_parameter("outputName"))
                    
            string_dict_data_s1Pre =  """{"satelliteParams": {"satelliteType": "Sentinel1","processingLevel": "LEVEL1","sensorMode": "IW","productType": "GRD"},"startDate":\"""" + str(w.get_parameter("s1_preFloodDate")) + """\","endDate": \"""" + str(w.get_parameter("floodDate")) + """\","geometry": \"""" + str(w.get_parameter("geometry")) + """\","area_sampling": "True","tmp_path_same_folder_dwl":"True","artifact_name":  \"""" + str(s1_artifact_pre) + """\"}"""
            string_dict_data_s1Post = """{"satelliteParams": {"satelliteType": "Sentinel1","processingLevel": "LEVEL1","sensorMode": "IW","productType": "GRD"},"startDate":\"""" + str(w.get_parameter("floodDate")) + """\","endDate": \"""" + str(w.get_parameter("s1_postFloodDate")) + """\","geometry": \"""" + str(w.get_parameter("geometry")) + """\","area_sampling": "True","tmp_path_same_folder_dwl":"True","artifact_name": \"""" + str(s1_artifact_post) + """\"}"""
            string_dict_data_s2Pre =  """{"satelliteParams":{"satelliteType": "Sentinel2","processingLevel": "S2MSI2A","bandmath": ["NDWI"]},"startDate":\"""" + str(w.get_parameter("s2_preFloodDate")) + """\","endDate": \"""" + str(w.get_parameter("floodDate")) + """\","geometry": \"""" + str(w.get_parameter("geometry")) + """\","cloudCover": "[0,20]","area_sampling": "True","artifact_name" : \"""" + str(s2_artifact_pre) + """\","preprocess_data_only": "false"}"""
            string_dict_data_s2Post = """{"satelliteParams":{"satelliteType": "Sentinel2","processingLevel": "S2MSI2A","bandmath": ["NDWI"]},"startDate":\"""" + str(w.get_parameter("floodDate")) + """\","endDate": \"""" + str(w.get_parameter("s2_postFloodDate")) + """\","geometry": \"""" + str(w.get_parameter("geometry")) + """\","cloudCover": "[0,20]","area_sampling": "True","artifact_name": \"""" + str(s2_artifact_post) + """\","preprocess_data_only": "false"}"""

            s1 = step(template={"action":"job",
                                "args":["main.py", string_dict_data_s1Pre],
                                "secrets":["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                                "fs_group":"8877",
                                "resources":{"mem":{"requests": "32Gi", "limits": "64Gi"}},
                                "volumes":[{"volume_type": "persistent_volume_claim","name": "volume-flood","mount_path": "/app/files","spec": { "size": "100Gi" }}]
                               }, 
                    function="download-images-s1",
                    name="s1-pre"
                   )
            
            s2 = step(template={"action":"job",
                                "args": ["main.py", string_dict_data_s1Post],
                                "secrets":["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                                "fs_group":"8877",
                                "resources":{"mem":{"requests": "32Gi", "limits": "64Gi"}},
                                "volumes":[{"volume_type": "persistent_volume_claim","name": "volume-flood","mount_path": "/app/files","spec": { "size": "100Gi" }}]
                               },
                    function="download-images-s1",
                    name="s1-post"
                    )
            
            s3 = step(template={"action":"job",
                                "args": ["main.py", string_dict_data_s2Pre],
                                "secrets":["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                                "fs_group":"8877",
                                "resources":{"mem":{"requests": "32Gi", "limits": "64Gi"}},
                                "volumes":[{"volume_type": "persistent_volume_claim","name": "volume-flood","mount_path": "/app/files","spec": { "size": "100Gi" }}]
                               },
                    function="download-images-s2",
                    name="s2-pre"
                    )
                    
            s4 = step(template={"action":"job",
                                "args": ["main.py", string_dict_data_s2Post],
                                "secrets":["CDSETOOL_ESA_USER","CDSETOOL_ESA_PASSWORD"],
                                "fs_group":"8877",
                                "resources":{"mem":{"requests": "32Gi", "limits": "64Gi"}},
                                "volumes":[{"volume_type": "persistent_volume_claim","name": "volume-flood","mount_path": "/app/files","spec": { "size": "100Gi" }}]
                               },
                    function="download-images-s2",
                    name="s2-post"
                    )
            
            s5 = step(template={"action":"job",
                                "args": ['/shared/launch.sh', str(s1_artifact_pre), str(s1_artifact_post), str(s2_artifact_pre), str(s2_artifact_post), str(w.get_parameter("geometry")), 'Slopes_TN', 'trentino_slope_map.tif', 'Lakes_TN', 'idrspacq.shp', 'Rivers_TN', 'cif_pta2022_v.shp', str(w.get_parameter("outputName")), str(w.get_parameter("floodDate")), 'EPSG:25832', "['VV','VH']", '900', '17', '9', '4', str(w.get_parameter("aoiName"))],
                                "fs_group":"8877",
                                "resources":{"cpu": {"requests": "3", "limits": "6"},"mem":{"requests": "32Gi", "limits": "64Gi"}},
                                "volumes":[{"volume_type": "persistent_volume_claim","name": "volume-flood","mount_path": "/app/data","spec": { "size": "200Gi" }}]
                               },
                    function="elaborate",
                    name="elaborate"
                    )
            
            s1 >> s2 >> s3 >> s4 >> s5

    return w

In [ ]:
workflow_hera = proj.new_workflow(name="pipeline_flood_hera", kind="hera", code_src="flood_pipeline_hera.py", handler="pipeline")

In [ ]:
wfbuild = workflow_hera.run(action="build", wait=True)

In [ ]:
workflow_hera_run = workflow_hera.run(action="pipeline", parameters={
    "geometry":"POLYGON ((11.123570406000056 45.89602853500003, 11.123570406000056 46.35080675000006, 11.833650907000049 46.35080675000006, 11.833650907000049 45.89602853500003, 11.123570406000056 45.89602853500003))",
    "outputName": "val_di_fassa_oct_2018",
    "floodDate":"2018-10-27",
    "aoiName": "val di fassa",
    "s1_preFloodDate": "2018-10-20",
    "s1_postFloodDate": "2018-11-04",
    "s2_preFloodDate": "2018-10-07",
    "s2_postFloodDate": "2018-11-17"
    })